# Chapter 4B: Fine-Tuning a Reasoning Text-to-SQL Model (Optional)

**Goal**: Take a model that outputs SQL directly and fine-tune it to *reason first*, then output SQL.

```
Before:  SELECT COUNT(*) FROM orders WHERE date > '2024-01-01'

After:   <reasoning>
         The question asks for order count since January 2024.
         I need the orders table with a date filter.
         </reasoning>
         <sql>SELECT COUNT(*) FROM orders WHERE date > '2024-01-01'</sql>
```

This `<reasoning>...<sql>` format is the same one used in Chapter 4C (GRPO). SFT teaches the **structure**; GRPO optimizes the **substance**.

- **Model**: `unsloth/Qwen2.5-Coder-1.5B-Instruct` (LoRA fine-tuning)
- **Dataset**: `gretelai/synthetic_text_to_sql` (105K examples with reasoning explanations)
- **Hardware**: Free Google Colab T4 GPU (~45-60 min training)
- **Standalone**: No dependency on the book's `src/` modules

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install sqlparse

## 2. Configuration

In [ ]:
# ============= CONFIGURATION =============

MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
DATASET_NAME = "gretelai/synthetic_text_to_sql"
TEST_SIZE = 200
MAX_SEQ_LENGTH = 2048

# LoRA — rank 16 is the sweet spot for domain adaptation tasks.
# Alpha = rank keeps scaling factor at 1 (stable default per Unsloth docs).
# Higher alpha (2x rank) increases adapter influence but risks instability.
LORA_R = 16
LORA_ALPHA = 16

# Training — 1 epoch is enough for 100K+ examples.
# Multi-epoch on large instruction datasets risks overfitting
# and can degrade results (Raschka 2024).
NUM_EPOCHS = 1
BATCH_SIZE = 2               # T4-friendly default; increase on larger GPUs
GRAD_ACCUM_STEPS = 16        # Effective batch = 32
LEARNING_RATE = 2e-4          # Standard for LoRA/QLoRA (Unsloth default)
WARMUP_RATIO = 0.05           # 5% of steps; more robust than fixed step count
WEIGHT_DECAY = 0.01

EVAL_EXAMPLES = 200
OUTPUT_DIR = "./output_4b_sft"

SYSTEM_PROMPT = (
    "You are a SQL expert. Given a database schema and a question, "
    "generate the correct SQL query. "
    "Respond using <reasoning> tags to explain your thinking step by step, "
    "then <sql> tags for the query."
)

## 3. GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected.")
    print("On Colab: Runtime > Change runtime type > T4 GPU")

## 4. Load Model + Attach LoRA Adapters

LoRA: base weights are frozen in float16, only the low-rank adapter matrices (~0.2%) are trainable. The base model's general SQL knowledge is preserved while the adapters learn domain-specific patterns.

> **Default in this notebook**: `load_in_4bit=False` (standard LoRA) on T4 for this 1.5B model, since it is often faster. If you hit memory limits or use larger models, switch to `load_in_4bit=True` (QLoRA).

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,      # LoRA default on T4 for 1.5B (often faster).
    dtype=None,               # Auto-detect: float16 or bfloat16
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Load Dataset

The `gretelai/synthetic_text_to_sql` dataset provides `(question, schema, SQL, explanation)` tuples. The `sql_explanation` field already contains natural language reasoning — no need to generate it ourselves.

We wrap the explanation in `<reasoning>` tags and the SQL in `<sql>` tags to teach the model our target format.

In [ ]:
import re
from datasets import load_dataset

# Load full dataset and rename columns for consistency
ds = load_dataset(DATASET_NAME, split="train")
ds = ds.rename_columns({
    "sql_prompt": "question",
    "sql_context": "context",
    "sql": "answer",
    "sql_explanation": "reasoning",
})
print(f"Full dataset: {len(ds):,} examples")

def is_read_only_sql(sql: str) -> bool:
    s = (sql or "").strip()
    if not s:
        return False
    first = re.match(r"^([a-zA-Z_]+)", s)
    first_kw = first.group(1).upper() if first else ""
    if first_kw not in {"SELECT", "WITH"}:
        return False

    # Block DML/DDL even if it appears inside CTEs
    forbidden = re.search(
        r"\\b(INSERT|UPDATE|DELETE|MERGE|UPSERT|DROP|CREATE|ALTER|TRUNCATE|REPLACE|GRANT|REVOKE)\\b",
        s,
        flags=re.IGNORECASE,
    )
    return forbidden is None

# Filter to read-only SQL (no DML/DDL)
before_n = len(ds)
ds = ds.filter(lambda x: is_read_only_sql(x["answer"]))
after_n = len(ds)
print(f"Read-only SQL subset: {after_n:,} / {before_n:,} examples")

# Split: hold out TEST_SIZE examples for evaluation, train on the rest
ds = ds.shuffle(seed=42)
split = ds.train_test_split(test_size=TEST_SIZE, seed=42)
train_ds_raw = split["train"]
test_ds_raw = split["test"]
print(f"Train: {len(train_ds_raw):,} | Test: {len(test_ds_raw):,}")

# Preview one example — note the reasoning comes from the dataset, not regex
ex = train_ds_raw[0]
print(f"\nQuestion: {ex['question']}")
print(f"SQL: {ex['answer']}")
print(f"\nDataset reasoning:\n{ex['reasoning']}")

In [ ]:
def format_cot_example(example):
    """Format as reasoning + SQL for SFT training."""
    assistant_content = (
        f"<reasoning>\n{example['reasoning']}\n</reasoning>\n"
        f"<sql>\n{example['answer']}\n</sql>"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Schema:\n{example['context']}\n\nQuestion: {example['question']}",
        },
        {"role": "assistant", "content": assistant_content},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        text = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{messages[1]['content']}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_content}<|im_end|>"
        )
    return {"text": text}


# Format all training examples
print("Generating CoT training data...")
train_formatted = train_ds_raw.map(
    format_cot_example, remove_columns=train_ds_raw.column_names
)
print(f"Formatted {len(train_formatted)} training examples.")

# Show one formatted example
print("\n--- Sample formatted example ---")
print(train_formatted[0]["text"][:800])
print("...")

## 6. Train

SFTTrainer teaches the model to produce `<reasoning>...</reasoning><sql>...</sql>` by imitating the training examples. Watch the loss decrease — starting ~1.5-2.0, ending ~0.5-0.8.

In [ ]:
import os
os.environ["NPROC"] = "1"           # Force single-process tokenization in Unsloth

import time
from trl import SFTTrainer
from transformers import TrainingArguments

total_steps = (len(train_formatted) // (BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print(f"Epochs: {NUM_EPOCHS}")
print(f"Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Training examples: {len(train_formatted):,}")
print(f"Estimated steps: ~{total_steps:,}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1,
    packing=True,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        logging_steps=50,
        save_steps=500,
        save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        optim="adamw_torch_fused",
        seed=42,
        report_to="none",
    ),
)

start_time = time.time()
train_result = trainer.train()
train_elapsed = time.time() - start_time

print(f"\nTraining complete in {train_elapsed/60:.1f} minutes")
print(f"Final training loss: {train_result.training_loss:.4f}")

## 7. Save LoRA Adapters

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to: {OUTPUT_DIR}")

## 7b. Continue Training on Saved Adapters

Already trained once and want to improve further? Load the saved adapters and resume training — with new data, more epochs, or different hyperparameters.

**What's actually saved in `./output_4b_sft/`:**

```
output_4b_sft/
├── adapter_config.json          # LoRA config + base model name
├── adapter_model.safetensors    # Only the A and B matrices (few MB)
├── tokenizer.json               # Tokenizer files
└── ...
```

The base model weights are **not** in this directory — only the small LoRA adapter matrices. The `adapter_config.json` contains a `base_model_name_or_path` field pointing to `unsloth/Qwen2.5-Coder-1.5B-Instruct`. When you load from this path, Unsloth reads that field, downloads the base model from HuggingFace (or loads from cache), then applies the adapters on top.

In [ ]:
# --- Continue training from saved adapters (run this in a new session) ---
# Uncomment and run this cell if you want to resume training.

"""
from unsloth import FastLanguageModel

# 1. Load base model + saved adapters
#    This directory contains only adapter weights (~few MB) + adapter_config.json.
#    adapter_config.json has "base_model_name_or_path" pointing to HuggingFace,
#    so Unsloth auto-downloads the base model and applies adapters on top.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./output_4b_sft",     # Adapter directory (not the full base model)
    max_seq_length=2048,
    load_in_4bit=False,     # Match your training mode
    dtype=None,
)

# 2. Prepare your new/additional dataset
# new_data = load_dataset("your_org/sql_examples", split="train")
# new_formatted = new_data.map(format_cot_example, ...)

# 3. Resume training with SFTTrainer (same as section 6)
#    Use a lower learning rate (e.g., 5e-5) for refinement
# trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=new_formatted, ...)
# trainer.train()

# 4. Save to a NEW path — never overwrite previous adapters.
#    This lets you compare versions and roll back if quality degrades.
# model.save_pretrained("./output_4b_sft_v2")
# tokenizer.save_pretrained("./output_4b_sft_v2")

# Typical versioning:
#   v1: gretelai public dataset          → ./output_4b_sft
#   v2: + your organization's SQL data   → ./output_4b_sft_v2
#   v3: + edge cases from production     → ./output_4b_sft_v3
"""

print("Section 7b: continue-training code is commented out.")
print("Uncomment when you want to resume training from saved adapters.")

## 8. Evaluate: Does the Model Now Reason?

We check three things:
1. **Format**: Does the output contain `<reasoning>` and `<sql>` tags?
2. **Exact match**: Does the normalized SQL match the gold answer exactly?
3. **Component score** (Spider-style): Compare SQL components (tables, SELECT, WHERE, GROUP BY, aggregations) individually. Two queries can differ in whitespace or alias naming but match on all components.

In [ ]:
import re
import sqlparse


def generate_output(model, tokenizer, schema: str, question: str):
    """Generate reasoning + SQL from schema + question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion: {question}"},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\nSchema:\n{schema}\n\nQuestion: {question}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,  # Deterministic decoding for reproducible evaluation
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def parse_output(text: str):
    """Extract reasoning and SQL from model output.

    Tries three formats in order:
    1. <reasoning>...</reasoning><sql>...</sql>  (our target format)
    2. ```sql ... ```  (markdown code block — base model often uses this)
    3. Raw text starting with SELECT/WITH  (direct SQL output)
    """
    # 1. Try our target format
    reasoning_match = re.search(r'<reasoning>(.*?)</reasoning>', text, re.DOTALL)
    sql_match = re.search(r'<sql>(.*?)</sql>', text, re.DOTALL)
    if reasoning_match and sql_match:
        return (
            reasoning_match.group(1).strip(),
            sql_match.group(1).strip(),
            True,  # has_format
        )

    # 2. Try markdown code block
    md_match = re.search(r'```(?:sql)?\s*(SELECT.*?|WITH.*?)```', text, re.DOTALL | re.IGNORECASE)
    if md_match:
        return None, md_match.group(1).strip(), False

    # 3. Try raw SQL at start
    raw_match = re.match(r'\s*(SELECT\b.*|WITH\b.*)', text, re.DOTALL | re.IGNORECASE)
    if raw_match:
        # Take first statement only
        sql = raw_match.group(1).split(';')[0].strip()
        return None, sql, False

    # 4. Fallback — return full text as SQL (will likely fail matching)
    return None, text.strip(), False


# --- Spider-style SQL comparison ---
# Instead of exact string match, normalize and compare SQL components.

def normalize_sql(sql: str) -> str:
    """Normalize SQL for comparison: lowercase, strip, collapse whitespace."""
    s = sql.strip().rstrip(";").lower()
    s = re.sub(r'\s+', ' ', s)           # collapse whitespace
    s = re.sub(r'\s*,\s*', ', ', s)      # normalize comma spacing
    s = re.sub(r'\s*\(\s*', '(', s)      # normalize paren spacing
    s = re.sub(r'\s*\)\s*', ')', s)
    return s.strip()


def extract_sql_components(sql: str) -> dict:
    """Extract key SQL components for Spider-style comparison."""
    sql = normalize_sql(sql)
    components = {}

    # Extract tables (FROM / JOIN clauses)
    tables = set()
    for m in re.finditer(r'\b(?:from|join)\s+(\w+)', sql):
        tables.add(m.group(1))
    components['tables'] = tables

    # Extract columns in SELECT
    select_match = re.match(r'select\s+(.*?)\s+from\b', sql, re.DOTALL)
    if select_match:
        components['select'] = normalize_sql(select_match.group(1))

    # Extract WHERE clause
    where_match = re.search(r'\bwhere\s+(.*?)(?:\bgroup\b|\border\b|\blimit\b|\bhaving\b|$)', sql, re.DOTALL)
    if where_match:
        components['where'] = normalize_sql(where_match.group(1))

    # Extract GROUP BY
    group_match = re.search(r'\bgroup\s+by\s+(.*?)(?:\bhaving\b|\border\b|\blimit\b|$)', sql, re.DOTALL)
    if group_match:
        components['group_by'] = normalize_sql(group_match.group(1))

    # Extract ORDER BY
    order_match = re.search(r'\border\s+by\s+(.*?)(?:\blimit\b|$)', sql, re.DOTALL)
    if order_match:
        components['order_by'] = normalize_sql(order_match.group(1))

    # Detect aggregations
    aggs = set(re.findall(r'\b(count|sum|avg|min|max)\s*\(', sql))
    if aggs:
        components['aggregations'] = aggs

    return components


def spider_style_match(pred_sql: str, gold_sql: str) -> dict:
    """Compare predicted vs gold SQL using Spider-style component matching.

    Returns dict with per-component match results and overall score.
    """
    pred = extract_sql_components(pred_sql)
    gold = extract_sql_components(gold_sql)

    results = {}
    matched = 0
    total = 0

    # Compare each component present in gold
    for key in gold:
        total += 1
        if key in pred and pred[key] == gold[key]:
            results[key] = 'match'
            matched += 1
        elif key in pred:
            results[key] = 'mismatch'
        else:
            results[key] = 'missing'

    # Exact normalized match as bonus check
    exact = normalize_sql(pred_sql) == normalize_sql(gold_sql)

    return {
        'component_results': results,
        'components_matched': matched,
        'components_total': total,
        'component_score': matched / total if total > 0 else 0.0,
        'exact_match': exact,
    }


print("Evaluation functions loaded (Spider-style component matching).")

In [ ]:
# Evaluate the fine-tuned model
FastLanguageModel.for_inference(model)

n = min(EVAL_EXAMPLES, len(test_ds_raw))
ft_results = []

print(f"Evaluating fine-tuned model on {n} test examples...\n")

for i in range(n):
    ex = test_ds_raw[i]
    output = generate_output(model, tokenizer, ex["context"], ex["question"])
    reasoning, sql, has_format = parse_output(output)
    gold = ex["answer"]
    match = spider_style_match(sql, gold)

    ft_results.append({
        "output": output, "reasoning": reasoning, "sql": sql,
        "gold": gold, "has_format": has_format, "match": match,
    })

    print(f"--- Example {i+1}/{n} ---")
    print(f"  Question:  {ex['question'][:80]}...")
    print(f"  Gold SQL:  {gold}")
    if reasoning:
        print(f"  Reasoning: {reasoning[:120]}{'...' if len(reasoning) > 120 else ''}")
    print(f"  Model SQL: {sql}")
    print(f"  Format OK:   {'YES' if has_format else 'no'}")
    print(f"  Exact match: {'YES' if match['exact_match'] else 'no'}")
    print(f"  Components:  {match['components_matched']}/{match['components_total']} ", end="")
    print(f"({match['component_score']:.0%})")
    for comp, status in match['component_results'].items():
        icon = 'OK' if status == 'match' else 'MISS' if status == 'missing' else 'DIFF'
        print(f"    {comp:<14} {icon}")
    print()

# Summary
format_ok = sum(1 for r in ft_results if r["has_format"])
exact_ok = sum(1 for r in ft_results if r["match"]["exact_match"])
avg_comp = sum(r["match"]["component_score"] for r in ft_results) / n if n else 0

print(f"{'='*50}")
print(f"  Format adherence:  {format_ok}/{n} ({100*format_ok/n:.0f}%)")
print(f"  Exact match:       {exact_ok}/{n} ({100*exact_ok/n:.0f}%)")
print(f"  Component score:   {avg_comp:.0%} (avg)")
print(f"{'='*50}")

## 9. Compare: Base Model vs Fine-Tuned

Load a fresh base model (no LoRA) to see the difference. The base model outputs SQL directly; the fine-tuned model should reason first.

In [ ]:
# Load fresh base model for comparison
print("Loading base model (without LoRA)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,       # 4-bit for comparison only (saves memory)
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

n = min(EVAL_EXAMPLES, len(test_ds_raw))
print(f"\nComparing on {n} examples...\n")

base_results = []

for i in range(n):
    ex = test_ds_raw[i]
    gold = ex["answer"]

    # Base model
    base_output = generate_output(base_model, base_tokenizer, ex["context"], ex["question"])
    b_reasoning, b_sql, b_has_format = parse_output(base_output)
    b_match = spider_style_match(b_sql, gold)
    base_results.append({"has_format": b_has_format, "match": b_match})

    # Fine-tuned (reuse cached results)
    f = ft_results[i]

    print(f"--- Example {i+1}/{n}: {ex['question'][:60]}... ---")
    print(f"  Base:       {base_output[:100]}...")
    print(f"  Fine-tuned: {f['output'][:100]}...")
    print(f"  Format:  base={'YES' if b_has_format else 'no':>3}  |  ft={'YES' if f['has_format'] else 'no':>3}")
    print(f"  Exact:   base={'YES' if b_match['exact_match'] else 'no':>3}  |  ft={'YES' if f['match']['exact_match'] else 'no':>3}")
    print(f"  CompScore: base={b_match['component_score']:.0%}  |  ft={f['match']['component_score']:.0%}")
    print()

# Free base model
del base_model, base_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Summary
b_fmt = sum(1 for r in base_results if r["has_format"])
f_fmt = sum(1 for r in ft_results if r["has_format"])
b_exact = sum(1 for r in base_results if r["match"]["exact_match"])
f_exact = sum(1 for r in ft_results if r["match"]["exact_match"])
b_comp = sum(r["match"]["component_score"] for r in base_results) / n
f_comp = sum(r["match"]["component_score"] for r in ft_results) / n

print(f"{'='*60}")
print(f"  {'Metric':<25} {'Base':<15} {'Fine-tuned':<15}")
print(f"  {'-'*25} {'-'*15} {'-'*15}")
print(f"  {'Format adherence':<25} {b_fmt}/{n} ({100*b_fmt/n:.0f}%){'':>5} {f_fmt}/{n} ({100*f_fmt/n:.0f}%)")
print(f"  {'Exact match':<25} {b_exact}/{n} ({100*b_exact/n:.0f}%){'':>5} {f_exact}/{n} ({100*f_exact/n:.0f}%)")
print(f"  {'Component score (avg)':<25} {b_comp:.0%}{'':>12} {f_comp:.0%}")
print(f"{'='*60}")

## 10. Summary

**What you did:**
- Took a model that outputs SQL directly (non-CoT)
- Used `gretelai/synthetic_text_to_sql` (105K examples with reasoning)
- Fine-tuned with LoRA + Unsloth SFT to produce `<reasoning>...<sql>` format
- The model now reasons before generating SQL

**Continue training:** Load saved adapters with `FastLanguageModel.from_pretrained("./output_4b_sft")` and resume training on new data or more epochs. Useful for iterative improvement — first train on public data, then refine on your organization's SQL examples.

**LoRA vs QLoRA:** On T4 for this 1.5B model, this notebook defaults to LoRA (`load_in_4bit=False`) because it is often faster. Use QLoRA (`load_in_4bit=True`) when memory is tight or for larger models.

**SFT ceiling:** Format adherence is high, but reasoning quality is bounded by training data.

**Serving the model in production:** Once trained, you need to serve the model for inference. Common options:

| Method | Use case |
|--------|----------|
| **vLLM** | High-throughput serving with continuous batching. Load base model + LoRA adapters. Supports hot-swapping multiple adapters on the same base model. |
| **llama.cpp (GGUF)** | CPU/edge inference. Merge adapters into base weights, then quantize to GGUF format. Single-file deployment. |
| **Merge + HuggingFace** | Merge adapters into base weights with `model.merge_and_unload()` for a standalone model. Simplest but loses the ability to swap adapters. |

Serving infrastructure and deployment patterns are covered in **Chapter 4C**.

**Next step — Chapter 4C:** GRPO uses reward functions (execution accuracy, reasoning quality) to push beyond the SFT imitation ceiling.